In [ ]:
import os
import requests
import json
from dotenv import load_dotenv
from typing import List

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.tools import tool
from pydantic import BaseModel, Field

import folium
from folium.plugins import AntPath

In [4]:
# Configuration
load_dotenv()
LLM_MODEL = "gemini-3.1-flash-lite"
google_api_key = os.getenv("GOOGLE_API_KEY")

In [5]:
URL_NET = "http://localhost:8000/network-awareness"
URL_LOC_UES = "http://localhost:8001/location/get-ues"
URL_LOC_GNBS = "http://localhost:8001/location/get-gnbs"
URL_ZONES = "http://localhost:8001/zones"

In [18]:
def get_processed_network_state():
    """
    Fetches real-time context from endpoints.
    Aborts immediately with an explicit error if endpoints are unreachable.
    """
    try:
        net_map = requests.get(URL_NET, timeout=2).json().get('radio_map', [])
        ue_locs_list = requests.get(URL_LOC_UES, timeout=2).json()
        gnb_locs_list = requests.get(URL_LOC_GNBS, timeout=2).json()
        zones = requests.get(URL_ZONES, timeout=2).json()
    except Exception as e:
        # CRITICAL: Stop execution loop immediately if APIs are down
        raise RuntimeError(
            f"Execution halted: Infrastructure endpoints are unreachable.\n"
            f"Details: {e}\n"
            f"Please ensure your localhost servers are running before planning routes."
        ) from e

    ue_lookup = {ue['ue_id']: ue for ue in ue_locs_list}
    gnb_lookup = {f"{gnb['id']:08d}": gnb for gnb in gnb_locs_list}

    processed_topology = {}
    for gnb in net_map:
        g_id = gnb['gnb_id']
        loc_info = gnb_lookup.get(g_id, {})
        
        active_ues = [
            {
                "supi": ue['supi'], 
                "lat": ue_lookup.get(ue['supi'], {}).get("lat", "N/A"), 
                "lng": ue_lookup.get(ue['supi'], {}).get("lng", "N/A")
            }
            for ue in gnb.get('connected_ues', [])
        ]
        
        processed_topology[f"gNodeB_{g_id}"] = {
            "lat": loc_info.get("lat", "N/A"),
            "lng": loc_info.get("lng", "N/A"),
            "radius": loc_info.get("radius", 0),
            "connected_ues": active_ues
        }

    return {
        "network_state": processed_topology, 
        "restricted_zones": zones, 
        "raw_gnbs": gnb_locs_list
    }

In [19]:
@tool
def get_flight_environment(query: str):
    """Returns current 5G network topology, gNB coverage, and restricted zones."""
    return get_processed_network_state()

In [20]:
class Waypoint(BaseModel):
    lat: float = Field(..., description="Latitude of the waypoint")
    lng: float = Field(..., description="Longitude of the waypoint")
    label: str = Field(..., description="Label like 'START', 'Waypoint 1', 'END'")

In [21]:
class FlightPlanResponse(BaseModel):
    reasoning: str = Field(..., description="Explanation of why this path was chosen relative to 5G coverage and Red zones.")
    waypoints: List[Waypoint] = Field(..., description="Ordered list of coordinates from START to END.")
    disclaimer: str = Field(..., description="Heuristic-based math validation warning.")

In [22]:
# Initialize LLM & Bind Structure
llm = ChatGoogleGenerativeAI(model=LLM_MODEL, temperature=0.1)
structured_llm = llm.with_structured_output(FlightPlanResponse)

In [23]:
# System Prompt
sys_msg = """You are a UAV Path Orchestrator.
Your task is to provide a safe list of waypoints from START to END based on tools data.
CRITICAL RULES:
1. Avoid RED ZONES at all costs.
2. Stay within the radius of gNodeBs to ensure signal (C2 Link).
3. If a direct path is impossible, provide intermediate waypoints.
4. Note that your coordinate math is 'heuristic-based'. State that a Graph-based Traversal (like A*) should validate your output in production.
"""

In [24]:
def run_uav_orchestrator(start: str, end: str):
    print(f"[Mission Start] Planning route for UAV...")
    
    # Fetch environment data (will raise exception if server is down)
    env_context = get_processed_network_state()
    
    prompt = f"""
    Context Data: {json.dumps(env_context)}
    Mission: Plan a safe path from {start} to {end}.
    Generate the structured JSON response.
    """
    
    response = structured_llm.invoke([SystemMessage(content=sys_msg), HumanMessage(content=prompt)])
    return response, env_context

In [25]:
# Execution Coordinates
start_coords = "23.235, 72.579"
end_coords = "23.246653, 72.603159"

output_data, environment = run_uav_orchestrator(start_coords, end_coords)

print("\n" + "="*40)
print("     UAV PATH RECOMMENDATION ENGINE (STRUCTURED)")
print("="*40)
print(f"Reasoning:\n{output_data.reasoning}\n")
print(f"Disclaimer: {output_data.disclaimer}\n")

[Mission Start] Planning route for UAV...

     UAV PATH RECOMMENDATION ENGINE (STRUCTURED)
Reasoning:
The direct path from the start point (23.235, 72.579) to the end point (23.246653, 72.603159) intersects with the airport-restricted red zone centered at (23.241357, 72.593231). To maintain C2 link connectivity, the path is routed through the coverage area of gNodeB_00000001 and gNodeB_00000002, bypassing the restricted zone to the north.

Disclaimer: This path was generated using heuristic-based coordinate calculations. A Graph-based Traversal (like A*) should validate this output in production to ensure absolute safety and optimal signal strength.



In [26]:
print("Waypoints:")
for wp in output_data.waypoints:
    print(f"  - {wp}")

Waypoints:
  - lat=23.235 lng=72.579 label='START'
  - lat=23.249099 lng=72.579446 label='Waypoint 1'
  - lat=23.255 lng=72.595 label='Waypoint 2'
  - lat=23.246653 lng=72.603159 label='END'


In [27]:
path_coordinates = [[wp.lat, wp.lng] for wp in output_data.waypoints]

# Base Map
m = folium.Map(
    location=path_coordinates[0], 
    zoom_start=14,
    tiles='https://{s}.basemaps.cartocdn.com/light_nolabels/{z}/{x}/{y}{r}.png',
    attr='CartoDB Positron No Labels'
)

# Plot Dynamically Retrieved gNB Circles
for gnb_key, gnb_data in environment["network_state"].items():
    if gnb_data["lat"] != "N/A":
        loc = [gnb_data["lat"], gnb_data["lng"]]
        folium.Circle(
            location=loc,
            radius=gnb_data["radius"],
            color="blue",
            fill=True,
            fill_opacity=0.10,
            popup=f"<b>{gnb_key}</b>"
        ).add_to(m)
        folium.Marker(
            location=loc,
            popup=f"<b>{gnb_key}</b>",
            icon=folium.Icon(color="blue", icon="signal")
        ).add_to(m)

# Plot Restricted Zones from API data
for zone in environment["restricted_zones"]:
    # Using defaults matching your setup if specific coords aren't returned by mock API
    zone_loc = [23.241357, 72.593231] 
    folium.Circle(
        location=zone_loc,
        radius=1000,
        color='red',
        fill=True,
        fill_opacity=0.35,
        popup=f"<b>RESTRICTED: {zone['zone_id']} ({zone['zone_type']})</b>"
    ).add_to(m)

# Plot the Dynamic Path
AntPath(
    locations=path_coordinates,
    color='black',
    weight=5,
    delay=1000,
    dash_array=[10, 20],
    pulse_color='white'
).add_to(m)

# Drop Dynamic Waypoint Markers
for wp in output_data.waypoints:
    if "START" in wp.label.upper():
        color, icon = "green", "play"
    elif "END" in wp.label.upper():
        color, icon = "red", "flag"
    else:
        color, icon = "orange", "info-sign"
        
    folium.Marker(
        location=[wp.lat, wp.lng],
        popup=wp.label,
        icon=folium.Icon(color=color, icon=icon)
    ).add_to(m)

In [28]:
m